In [1]:
!pip install wandb -q

In [2]:
import wandb

In [4]:
wandb.login(relogin=True)

<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [6]:
import numpy as np
import pandas as pd
import os

def generate_dataset(seed, num_samples=1000):
    np.random.seed(seed)
    X = np.random.randn(num_samples, 5)  # 5 features
    # We'll create a binary target that somewhat depends on the first two features
    y = (X[:, 0] + 0.5 * X[:, 1] + 0.1 * np.random.randn(num_samples) > 0).astype(int)
    return X, y

In [7]:
X_test, y_test = generate_dataset(seed=42, num_samples=200)

In [8]:
run = wandb.init(project="my-gita-wandb", name="test_artifact_creation",
                 notes="Create a fixed test dataset artifact")

wandb: Currently logged in as: mordy. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


In [9]:
df_test = pd.DataFrame(X_test, columns=[f"feature_{i}" for i in range(X_test.shape[1])])
df_test["target"] = y_test

In [10]:
test_filename = "test_data.csv"
df_test.to_csv(test_filename, index=False)

In [11]:
test_artifact = wandb.Artifact("my_dataset", type="dataset", description="Fixed test set")
test_artifact.add_file(test_filename)
run.log_artifact(test_artifact)

<Artifact my_dataset>

In [12]:
run.finish()

In [13]:
train_seeds = [101, 202]

for version, seed in enumerate(train_seeds):
    run = wandb.init(project="my-gita-wandb",
                     name=f"train_set_v{version}",
                     notes=f"Create training set version {version}")

    # Generate new random dataset for training
    X_train, y_train = generate_dataset(seed=seed, num_samples=1000)
    df_train = pd.DataFrame(X_train, columns=[f"feature_{i}" for i in range(X_train.shape[1])])
    df_train["target"] = y_train

    # Save to CSV
    train_filename = f"train_data_v{version}.csv"
    df_train.to_csv(train_filename, index=False)

    # Create & log artifact
    artifact = wandb.Artifact("train_data",
                              type="dataset",
                              description=f"Training dataset version {version}")
    artifact.add_file(train_filename)
    run.log_artifact(artifact)

    run.finish()


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import os

train_versions = [0, 1]  # We'll use 'train_data:v0' and 'train_data:v1'

for version_number in train_versions:
    run = wandb.init(project="my-gita-wandb",
                     name=f"train_model_with_data_v{version_number}",
                     notes=f"Training a model using train_data:v{version_number}")

    # 1. Download the train data artifact
    artifact = run.use_artifact(f"train_data:v{version_number}", type="dataset")
    artifact_dir = artifact.download()
    train_csv = os.path.join(artifact_dir, f"train_data_v{version_number}.csv")
    df_train = pd.read_csv(train_csv)
    X_train = df_train.drop("target", axis=1).values
    y_train = df_train["target"].values

    # 2. Download the fixed test set artifact
    test_art = run.use_artifact("my_dataset:latest", type="dataset")
    test_dir = test_art.download()
    test_csv = os.path.join(test_dir, "test_data.csv")
    df_test = pd.read_csv(test_csv)
    X_test = df_test.drop("target", axis=1).values
    y_test = df_test["target"].values

    # 3. Train a simple Logistic Regression model
    model = LogisticRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # 4. Calculate and log our metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Log scalar metrics
    wandb.log({"accuracy": acc, "f1_score": f1})

    # Optionally log a confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_test,
        preds=y_pred,
        class_names=["Class 0", "Class 1"]
    )})

    run.finish()

wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


accuracy,▁
f1_score,▁
accuracy,0.975
f1_score,0.9763


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


accuracy,▁
f1_score,▁
accuracy,0.97
f1_score,0.97196
